In [ ]:
import numpy as np
import pandas as pd
import os
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [ ]:
data_dir = '/kaggle/input/prostate-cancer-grade-assessment/train_images'
labels_path = '/kaggle/input/prostate-cancer-grade-assessment/train.csv'
labels_df = pd.read_csv(labels_path)

In [ ]:
subset_size = 10
subset_df = labels_df.sample(n=subset_size, random_state=42)

In [ ]:
img_size = 128
X = []
y = []

Image.MAX_IMAGE_PIXELS = None

for idx, row in subset_df.iterrows():
    img_path = os.path.join(data_dir, row['image_id'] + '.tiff')
    if os.path.exists(img_path):
        img = Image.open(img_path)
        img = img.resize((img_size, img_size))
        img = np.array(img)
        X.append(img)
        y.append(row['isup_grade'])

X = np.array(X)
y = np.array(y)

num_classes = 6 
y = to_categorical(y, num_classes=num_classes)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_size, img_size, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax') 
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.show()

model.save('prostate_cancer_model.h5')